In [0]:

dbutils.widgets.text("environment", "dev")
env = dbutils.widgets.get("environment")
print(env)

In [0]:
spark.sql(f"drop table if exists {env}_banking.metadata.tables")
spark.sql(f"drop table if exists {env}_banking.metadata.table_parameters")
spark.sql(f"drop table if exists {env}_banking.metadata.table_watermarks")
spark.sql(f"drop table if exists {env}_banking.metadata.pipeline_runs")
spark.sql(f"drop table if exists {env}_banking.metadata.batch_control")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.metadata.batch_control (
    ingestion_run_id STRING,
    batch_id STRING,
    source_table STRING,
    target_table STRING,
    target_layer STRING,
    load_type STRING,
    status STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    record_count BIGINT,
    error_message STRING
) USING DELTA        
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.metadata.tables (
    table_id            INT,
    table_name          STRING,        -- table name (target table)
    source_system       STRING,        -- sqlserver / blob
    source_schema       STRING,        -- dbo (null for blob)
    source_table        STRING,        -- table name (null for blob)
    source_path         STRING,        -- blob path (null for sqlserver)
    target_layer        STRING,        -- silver/gold
    active_flag         BOOLEAN,
    load_order          INT,
    created_at          TIMESTAMP
)
USING DELTA;
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.metadata.table_parameters (
    table_id            INT,
    parameter_name      STRING,        -- load_type / primary_key / watermark_column
    parameter_value     STRING,
    created_at          TIMESTAMP
)
USING DELTA;
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.metadata.table_watermarks (
    process_name            STRING,
    table_name              STRING,
    last_source_timestamp   TIMESTAMP,     -- flexible type storage
    last_ingestion_run_id   STRING,
    last_updated_at         TIMESTAMP
)
USING DELTA
""")

In [0]:
spark.sql(f"""
    INSERT INTO {env}_banking.metadata.table_watermarks VALUES
    ('bronze', 'customer_history', '1900-01-01 00:00:00', NULL, current_timestamp()),
    ('bronze', 'transaction', '1900-01-01 00:00:00', NULL, current_timestamp()),
    ('bronze', 'account', '1900-01-01 00:00:00', NULL, current_timestamp()),
    ('bronze', 'account_customer_history', '1900-01-01 00:00:00', NULL, current_timestamp()),
    ('bronze', 'transaction_line', '1900-01-01 00:00:00', NULL, current_timestamp()),
    ('bronze', 'daily_account_balance', '1900-01-01 00:00:00', NULL, current_timestamp())
""")

spark.sql(f"""
    INSERT INTO {env}_banking.metadata.table_watermarks VALUES
    ('silver', 'customer_history', '1900-01-01 00:00:00', NULL, current_timestamp()),
    ('silver', 'transaction', '1900-01-01 00:00:00', NULL, current_timestamp()),
    ('silver', 'account', '1900-01-01 00:00:00', NULL, current_timestamp()),
    ('silver', 'account_customer_history', '1900-01-01 00:00:00', NULL, current_timestamp()),
    ('silver', 'transaction_line', '1900-01-01 00:00:00', NULL, current_timestamp()),
    ('silver', 'daily_account_balance', '1900-01-01 00:00:00', NULL, current_timestamp())
""")

spark.sql(f"""
    INSERT INTO {env}_banking.metadata.table_watermarks VALUES
        ('ods', 'customer_history', '1900-01-01 00:00:00', NULL, current_timestamp()),
        ('ods', 'transaction', '1900-01-01 00:00:00', NULL, current_timestamp()),
        ('ods', 'account', '1900-01-01 00:00:00', NULL, current_timestamp()),
        ('ods', 'account_customer_history', '1900-01-01 00:00:00', NULL, current_timestamp()),
        ('ods', 'transaction_line', '1900-01-01 00:00:00', NULL, current_timestamp()),
        ('ods', 'daily_account_balance', '1900-01-01 00:00:00', NULL, current_timestamp())
""")


In [0]:
spark.sql(f"""
   INSERT INTO {env}_banking.metadata.tables VALUES
    (1, 'branch', 'sqlserver', 'data_source', 'branch', NULL, 'bronze',  TRUE, 1, current_timestamp()),
    (2, 'customer_history', 'sqlserver', 'data_source', 'customer_history', NULL, 'bronze', TRUE, 2, current_timestamp()),
    (3, 'transaction', 'sqlserver', 'data_source', 'transaction', NULL, 'bronze', TRUE, 3, current_timestamp()),
    (4, 'account', 'sqlserver', 'data_source', 'account', NULL, 'bronze', TRUE, 4, current_timestamp()),
    (5, 'product', 'sqlserver', 'data_source', 'product', NULL, 'bronze', TRUE, 5, current_timestamp()),
    (6, 'account_customer_history', 'sqlserver', 'data_source', 'account_customer_history', NULL, 'bronze', TRUE, 6, current_timestamp()),
    (7, 'transaction_line', 'sqlserver', 'data_source', 'transaction_line', NULL, 'bronze', TRUE, 7, current_timestamp()),
    (8, 'daily_account_balance', 'sqlserver', 'data_source', 'daily_account_balance', NULL, 'bronze', TRUE, 8, current_timestamp())
""")

spark.sql(f"""
   INSERT INTO {env}_banking.metadata.tables VALUES
    (10, 'branch', 'delta', 'bronze', 'branch', NULL, 'silver',  TRUE, 1, current_timestamp()),
    (11, 'customer_history', 'delta', 'bronze', 'customer_history', NULL, 'silver', TRUE, 2, current_timestamp()),
    (12, 'transaction', 'delta', 'bronze', 'transaction', NULL, 'silver', TRUE, 3, current_timestamp()),
    (13, 'account', 'delta', 'bronze', 'account', NULL, 'silver', TRUE, 4, current_timestamp()),
    (14, 'product', 'delta', 'bronze', 'product', NULL, 'silver', TRUE, 5, current_timestamp()),
    (15, 'account_customer_history', 'delta', 'bronze', 'account_customer_history', NULL, 'silver', TRUE, 6, current_timestamp()),
    (16, 'transaction_line', 'delta', 'bronze', 'transaction_line', NULL, 'silver', TRUE, 7, current_timestamp()),
    (17, 'daily_account_balance', 'delta', 'bronze', 'daily_account_balance', NULL, 'silver', TRUE, 8, current_timestamp())
""")

spark.sql(f"""
   INSERT INTO {env}_banking.metadata.tables VALUES
    (20, 'branch', 'delta', 'silver', 'branch', NULL, 'ods',  TRUE, 1, current_timestamp()),
    (21, 'customer_history', 'delta', 'silver', 'customer_history', NULL, 'ods', TRUE, 2, current_timestamp()),
    (22, 'transaction', 'delta', 'silver', 'transaction', NULL, 'ods', TRUE, 3, current_timestamp()),
    (23, 'account', 'delta', 'silver', 'account', NULL, 'ods', TRUE, 4, current_timestamp()),
    (24, 'product', 'delta', 'silver', 'product', NULL, 'ods', TRUE, 5, current_timestamp()),
    (25, 'account_customer_history', 'delta', 'silver', 'account_customer_history', NULL, 'ods', TRUE, 6, current_timestamp()),
    (26, 'transaction_line', 'delta', 'silver', 'transaction_line', NULL, 'ods', TRUE, 7, current_timestamp()),
    (27, 'daily_account_balance', 'delta', 'silver', 'daily_account_balance', NULL, 'ods', TRUE, 8, current_timestamp())
""")


In [0]:
spark.sql(f"SELECT * FROM {env}_banking.metadata.tables").show()

In [0]:
%sql
USE IDENTIFIER(:environment || '_banking.metadata')

In [0]:
%sql
INSERT INTO table_parameters VALUES

-- ================= CUSTOMERS HISTORY =================
(2, 'load_type', 'MERGE', current_timestamp()),
(2, 'primary_key', 'customer_id,effective_from_date', current_timestamp()),
(2, 'watermark_column', 'source_modified_at', current_timestamp()),

-- ================= TRANSACTIONS =================
(3, 'load_type', 'APPEND', current_timestamp()),
(3, 'primary_key', 'transaction_id', current_timestamp()),
(3, 'watermark_column', 'source_modified_at', current_timestamp()),

-- ================= ACCOUNTS =================
(4, 'load_type', 'MERGE', current_timestamp()),
(4, 'primary_key', 'account_id', current_timestamp()),
(4, 'watermark_column', 'source_modified_at', current_timestamp()),

-- ================= BRANCH =================
(1, 'load_type', 'FULL', current_timestamp()),
(1, 'primary_key', 'branch_id', current_timestamp()),

-- ================= PRODUCT =================
(5, 'load_type', 'FULL', current_timestamp()),
(5, 'primary_key', 'product_id', current_timestamp()),

-- ================= ACCOUNT CUSTOMERS HISTORY =================
(6, 'load_type', 'MERGE', current_timestamp()),
(6, 'primary_key', 'account_id,customer_id,effective_from_date', current_timestamp()),
(6, 'watermark_column', 'source_modified_at', current_timestamp()),

--  ================= TRANSACTIONS LINE=================
(7, 'load_type', 'APPEND', current_timestamp()),
(7, 'primary_key', 'transaction_id,line_number', current_timestamp()),
(7, 'watermark_column', 'source_modified_at', current_timestamp()),

--  ================= DAILY ACCOUNT BALANCE =================
(8, 'load_type', 'MERGE', current_timestamp()),
(8, 'primary_key', 'balance_date,account_id', current_timestamp()),
(8, 'watermark_column', 'source_modified_at', current_timestamp());

In [0]:
%sql
INSERT INTO table_parameters VALUES

-- ================= CUSTOMERS HISTORY =================
(11, 'load_type', 'MERGE', current_timestamp()),
(11, 'primary_key', 'customer_id,effective_from_date', current_timestamp()),
(11, 'watermark_column', 'source_modified_at', current_timestamp()),

-- ================= TRANSACTIONS =================
(12, 'load_type', 'APPEND', current_timestamp()),
(12, 'primary_key', 'transaction_id', current_timestamp()),
(12, 'watermark_column', 'source_modified_at', current_timestamp()),

-- ================= ACCOUNTS =================
(13, 'load_type', 'MERGE', current_timestamp()),
(13, 'primary_key', 'account_id', current_timestamp()),
(13, 'watermark_column', 'source_modified_at', current_timestamp()),

-- ================= BRANCH =================
(10, 'load_type', 'FULL', current_timestamp()),
(10, 'primary_key', 'branch_id', current_timestamp()),

-- ================= PRODUCT =================
(14, 'load_type', 'FULL', current_timestamp()),
(14, 'primary_key', 'product_id', current_timestamp()),

-- ================= ACCOUNT CUSTOMERS HISTORY =================
(15, 'load_type', 'MERGE', current_timestamp()),
(15, 'primary_key', 'account_id,customer_id,effective_from_date', current_timestamp()),
(15, 'watermark_column', 'source_modified_at', current_timestamp()),

--  ================= TRANSACTIONS LINE=================
(16, 'load_type', 'APPEND', current_timestamp()),
(16, 'primary_key', 'transaction_id,line_number', current_timestamp()),
(16, 'watermark_column', 'source_modified_at', current_timestamp()),

--  ================= DAILY ACCOUNT BALANCE =================
(17, 'load_type', 'MERGE', current_timestamp()),
(17, 'primary_key', 'balance_date,account_id', current_timestamp()),
(17, 'watermark_column', 'source_modified_at', current_timestamp());

In [0]:
%sql
INSERT INTO table_parameters VALUES

-- ================= CUSTOMERS HISTORY =================
(21, 'load_type', 'MERGE', current_timestamp()),
(21, 'primary_key', 'customer_id,effective_from_date', current_timestamp()),
(21, 'watermark_column', 'source_modified_at', current_timestamp()),

-- ================= TRANSACTIONS =================
(22, 'load_type', 'APPEND', current_timestamp()),
(22, 'primary_key', 'transaction_id', current_timestamp()),
(22, 'watermark_column', 'source_modified_at', current_timestamp()),

-- ================= ACCOUNTS =================
(23, 'load_type', 'MERGE', current_timestamp()),
(23, 'primary_key', 'account_id', current_timestamp()),
(23, 'watermark_column', 'source_modified_at', current_timestamp()),

-- ================= BRANCH =================
(20, 'load_type', 'FULL', current_timestamp()),
(20, 'primary_key', 'branch_id', current_timestamp()),

-- ================= PRODUCT =================
(24, 'load_type', 'FULL', current_timestamp()),
(24, 'primary_key', 'product_id', current_timestamp()),

-- ================= ACCOUNT CUSTOMERS HISTORY =================
(25, 'load_type', 'MERGE', current_timestamp()),
(25, 'primary_key', 'account_id,customer_id,effective_from_date', current_timestamp()),
(25, 'watermark_column', 'source_modified_at', current_timestamp()),

--  ================= TRANSACTIONS LINE=================
(26, 'load_type', 'APPEND', current_timestamp()),
(26, 'primary_key', 'transaction_id,line_number', current_timestamp()),
(26, 'watermark_column', 'source_modified_at', current_timestamp()),

--  ================= DAILY ACCOUNT BALANCE =================
(27, 'load_type', 'MERGE', current_timestamp()),
(27, 'primary_key', 'balance_date,account_id', current_timestamp()),
(27, 'watermark_column', 'source_modified_at', current_timestamp());

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.metadata.ods_incremental_work_table (
    customer_id STRING,
    account_id STRING
) USING DELTA        
""")